# Characterizing the difference between Eastern Oyster transcriptomes in a New York versus Virginia Hatchery

Hello, this little project is based off data from the publication: "Contrasting Immunomodulatory Effects of Probiotic and Pathogenic Bacteria on Eastern Oyster, Crassostrea Virginica, Larvae", doi: 10.3390/vaccines8040588 . This publication compares the transcriptome of different oysters when treated with various probiotics aimed at making them more resistant to a parasitic disease. 

This project will only use the control data 

First we will begin by downloading the raw reads via the SRA toolkit

In [6]:
system("ls", intern = TRUE)
library("IRdisplay")
library(Rhisat2)
library(rtracklayer)

[1] "SRR10982729_1.fastq"                "SRR10982729_2.fastq"               
[3] "eastern_oyster_transcriptome.ipynb"

Now we will fetch our data via the SRA toolkit. 

In [1]:
SRA_list <- c("SRR14150616",  # New Brunswick 1
              "SRR14150615",  # New Brunswick 2  
              "SRR14150614",  # New Brunswick 3
              "SRR14150613",  # New Brunswick 4
              "SRR14150617",  # Louisiana 4
              "SRR14150618",  # Louisiana 3
              "SRR14150619",  # Louisiana 2
              "SRR14150620")  # Louisiana 1

We will prefetch all of our data from the NCBI database. This will mkae the download go much faster than just fasterq dumping everything by dividing the downloading and processing steps. Note that this will use up more temporary space. 

In [4]:
system("vdb-config --prefetch-to-cwd")


for (SRR in SRA_list) {
    system(paste0("prefetch ", SRR))
    }


Now we will do the fasterq dump to get our actual fastq files

In [5]:
for (SRR in SRA_list) {
    system(paste0("fasterq-dump ", SRR))
    }

In [6]:
# making a dedictaed directory 
system("mkdir fastq_files")
system("mv *.fastq fastq_files/")


Now we will run some fastqc in order to do some quality checks on our data.

In [7]:
system("fastqc fastq_files/*.fastq")

#make a directory for the new fastqc files
system("mkdir qc_reports")
system("mv fastq_files/*fastqc.* qc_reports/")

It takes a bit too long to inspect every fastqc files. Using multqc to generate a summary of all fastqc reports is the best way to visualize the overall quality of our data.

In [10]:
# conducts the summary of all theqc reports and moves the  files so that they are the 
system("multiqc qc_reports/")

In [4]:
# system2("conda", args = c("install", "r-irdisplay"))

In [ ]:
library("IRdisplay")
display_html(file = "multiqc_report.html")


## Trimming the Data

Now we will be using the fastp command in order to trim the reads which contain a large amount of adapter sequences. In order to do iterate through all our files we will be using a bash script which I will post below for your convenience:


#!/bin/bash
mkdir -p trimmed_data
cd fastq_files
for r1 in *_1.fastq; do
  r2="${r1%_1.fastq}_2.fastq"
  fastp -i "$r1" -I "$r2" \
        -o "~/e_oyster/project_2/trimmed_data/trimmed_${r1}" -O "~/e_oyster/project_2/trimmed_data/trimmed_${r2}" \
        --html "${r1%.fastq}.html" 
done

In [13]:
system2("conda", args=c("install -c bioconda fastp"))


In [17]:
# This will execute our bash script
system("bash trim_reads.sh")

In [16]:
system("pwd")

In [18]:
system2("fastqc", args= c("trimmed_data/*.fastq"))

In [23]:
# Moving the fastqc reports to the qc reports folder
system2("mkdir", args= c("-p qc_reports/trimmed_reports"))
system2("mv", args= c("raw_data/trimmed_data/*fastqc.*","qc_reports/trimmed_reports/"))

In [24]:
system2("multiqc", args = c("qc_reports/trimmed_reports/"))

Now we will delete the old fastq files to save on some storage space but we will keep the fastqc reports

In [2]:
# be careful with this command
system("rm -r fastq_files")

[View the trimmed multiqc report HERE](multiqc_report_1.html)


If you view the multiqc report, then you will see that the qc report indicates that all of the adapter sequences have been removed from our files. Now we can proceed with our analysis

## HISAT2 alignement

Now that we have our qc's data, we will need to align our transcript data. Since I am conducting this analysis on modest, local hardware I'll use the HISAT2 aligner. If I were doing this analysis on the cloud or on an HPC then I would use STAR for its multithreading capabilities instead.

In [10]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
system2("conda", args = c("install -c bioconda bioconductor-Rhisat2"))


In [2]:
system2("datasets", args= c("download genome taxon Crassostrea virginica --reference --include genome,gff3,rna"))
system2("unzip", args= c("ncbi_dataset.zip"))


In [1]:
library(Rhisat2)

# Build the index from reference fasta files
hisat2_build(
    references = "ncbi_dataset/data/GCF_053477285.1/GCF_053477285.1_ASM5347728v1_genomic.fna", 
    outdir = "ref_seq", 
    prefix = "oyster_index"
)


In [14]:
library(GenomicFeatures)

txdb <- makeTxDbFromGFF("ncbi_dataset/data/GCF_053477285.1/genomic.gff", format="gff3")

Loading required package: AnnotationDbi

Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .extract_exons_from_GRanges(exon_IDX, gr, mcols0, tx_IDX, feature = "exon", :
“23115 exons couldn't be linked to a transcript so were dropped (showing
  only the first 6):
        seqid  start    end strand                    ID                  Name
1 NC_136047.1 211526 213254      + exon-XR_013560623.1-1 exon-XR_013560623.1-1
2 NC_136047.1 214781 215178      + exon-XR_013560623.1-2 exon-XR_013560623.1-2
3 NC_136047.1 568106 568195      + exon-XR_013558991.1-1 exon-XR_013558991.1-1
4 NC_136047.1 573790 574101      + exon-XR_013558991.1-2 exon-XR_013558991.1-2
5 

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'rownames': unable to find an inherited method for function ‘mcols’ for signature ‘"TxDb"’


In [14]:
library(rtracklayer)
library(GenomicFeatures)
library(Rhisat2)

Loading required package: AnnotationDbi

Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.




In [2]:
# Ensure you have 'gffread' installed on your system
# system(" conda install -c bioconda gffread ")

# converts the gff file in to a gtf file for hisat2
system("gffread ncbi_dataset/data/GCF_053477285.1/genomic.gff -F -T -o ncbi_dataset/data/GCF_053477285.1/converted_genomic.gtf")

In [3]:
# imports the newly converted file
gr <- import("ncbi_dataset/data/GCF_053477285.1/converted_genomic.gtf")


In [25]:
system("head ncbi_dataset/data/GCF_053477285.1/converted_genomic.gtf")

In [21]:
na_trans <- is.na(gr$transcript_id) & !is.na(gr$ID)
head(gr$ID[na_trans])

NULL

In [10]:
faulty_ind <- which(is.na(gr$gene_id))
# show why the gene_id must be replaced with the gene name
mcols(gr)[faulty_ind[4],"gene_name"]
mcols(gr)[faulty_ind[4],"transcript_id"]

mcols(gr)[faulty_ind, "gene_id"] <- mcols(gr)[faulty_ind, "gene_name"] 

[1] "LOC144618302"

[1] "gene-LOC144618302"

In [11]:
# 2. Build the clean TxDb object
txdb <- makeTxDbFromGRanges(gr)

In [ ]:
# lists splice sites for the alignment process
extract_splice_sites(features = txdb, outfile = "splice_sites.txt")

# now we will re export the edited gr object so that we can use the same annotations for when we do feature counts
export(gr,  con = "ncbi_dataset/data/GCF_053477285.1/edited_genomic.gtf", format = "gtf")




In [31]:
# checks
# length(which(is.na(mcols(gr)$gene_id)))
# gr <- gr[!which(is.na(mcols(gr)$gene_id))]

[1] 2412

In [33]:
gr <- gr[!which(is.na(mcols(gr)$gene_id))]

In [12]:
# this will pair together the forward and backward reads files together in a list

r1_files <- list.files(path = "trimmed_data",  pattern = "_1.*\\.(fastq)$", full.names = TRUE)
r2_files <- list.files(path = "trimmed_data", pattern = "_2.*\\.(fastq)$", full.names = TRUE)

r1_files <- sort(r1_files)
r2_files <- sort(r2_files)

paired_files <- list()
for (i in 1:length(r1_files)){
    paired_files[[i]] <-  list(r1_files[i], r2_files[i])
    }
paired_files


[[1]]
[[1]][[1]]
[1] "trimmed_data/trimmed_SRR14150620_1.fastq"

[[1]][[2]]
[1] "trimmed_data/trimmed_SRR14150620_2.fastq"

character(0)

In [15]:
for (path in paired_files) {
    cur_SRR <- tools::file_path_sans_ext(basename(path[[1]]))
    
    hisat2(
        sequences = path, 
        index = file.path("ref_seq", "oyster_index"), 
        type = "paired", 
        outfile = paste0("aligned_", cur_SRR, ".sam"),
        force = TRUE,
        strict = TRUE,
        execute = TRUE,
        `known-splicesite-infile` = "splice_sites.txt",
        p = 4  # number of threads
    )
    }

In [1]:
file.path("ref_seq", "oyster_index")

[1] "ref_seq/oyster_index"

In [4]:
library(Rhisat2)

In [7]:
file.path("ref_seq", "oyster_index")

[1] "ref_seq/oyster_index"

In [8]:
path

[[1]]
[1] "raw_data/trimmed_data/trimmed_SRR10982743_1.fastq"

[[2]]
[1] "raw_data/trimmed_data/trimmed_SRR10982743_2.fastq"

Now that we have all of our  sam files, we must have them as an indexed bam file. This way they can also be viewed in genomic editors such as UCSC genome browser.

In [1]:
library(Rsamtools)
sam_files <- list.files(path = ".", pattern = "\\.sam$")


for (f in sam_files) {
    dest <- paste0("bam_", tools::file_path_sans_ext(f))
    asBam(f, overwrite = TRUE, destination = dest )
}



Loading required package: GenomeInfoDb

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loa

ERROR: Error in value[[3L]](cond): 'asBam' truncated input file at record 32972325
  SAM file: 'aligned_trimmed_SRR14150620_1.sam'



In [16]:

asBam('aligned_trimmed_SRR14150620_1.sam', dest = 'bam_aligned_trimmed_SRR14150620_1')

[1] "bam_aligned_trimmed_SRR14150620_1.bam"

In [17]:
system("mkdir sorted_bam_files")
system("mv *.bam* sorted_bam_files/")


In [2]:

bam_files <- list.files(path = "sorted_bam_files", pattern = "\\.bam$", full.names = TRUE)
print(bam_files)

[1] "sorted_bam_files/bam_aligned_trimmed_SRR14150613_1.bam"
[2] "sorted_bam_files/bam_aligned_trimmed_SRR14150614_1.bam"
[3] "sorted_bam_files/bam_aligned_trimmed_SRR14150615_1.bam"
[4] "sorted_bam_files/bam_aligned_trimmed_SRR14150616_1.bam"
[5] "sorted_bam_files/bam_aligned_trimmed_SRR14150617_1.bam"
[6] "sorted_bam_files/bam_aligned_trimmed_SRR14150618_1.bam"
[7] "sorted_bam_files/bam_aligned_trimmed_SRR14150619_1.bam"
[8] "sorted_bam_files/bam_aligned_trimmed_SRR14150620_1.bam"


In [3]:
library(Rsubread)

counts <- featureCounts(
  files = bam_files, 
  annot.ext = "ncbi_dataset/data/GCF_053477285.1/edited_genomic.gtf", 
  isGTFAnnotationFile = TRUE,
  GTF.featureType = "exon",      # Level to count (usually exon)
  GTF.attrType = "gene_id",      # Attribute to aggregate by (usually gene_id)
  isPairedEnd = TRUE,            
  nthreads = 8
)

# Extract the count matrix
#gene_counts <- counts$counts


        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
       Rsubread 2.16.1

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 8 BAM files                                      ||
||                                                                            ||
||                           bam_aligned_trimmed_SRR14150613_1.bam            ||
||                           bam_aligned_trimmed_SRR14150614_1.bam            ||
||                           bam_aligned_trimmed_SRR

Rsubread


In [8]:
system("conda install -c bioconda samtools")

In [4]:
gene_counts <- counts$counts
print(colnames(gene_counts))

[1] "bam_aligned_trimmed_SRR14150613_1.bam"
[2] "bam_aligned_trimmed_SRR14150614_1.bam"
[3] "bam_aligned_trimmed_SRR14150615_1.bam"
[4] "bam_aligned_trimmed_SRR14150616_1.bam"
[5] "bam_aligned_trimmed_SRR14150617_1.bam"
[6] "bam_aligned_trimmed_SRR14150618_1.bam"
[7] "bam_aligned_trimmed_SRR14150619_1.bam"
[8] "bam_aligned_trimmed_SRR14150620_1.bam"


In [44]:
system("conda install bioconda::bioconductor-edgeR")

[1] "New_Brunswick" "New_Brunswick" "New_Brunswick" "New_Brunswick"
[5] "Louisiana"     "Louisiana"     "Louisiana"     "Louisiana"

In [7]:
library(edgeR)
# Assume 'counts' is a numeric matrix (genes in rows, samples in columns)
# Assume 'group' is a factor or vector indicating conditions (e.g., c("control", "control", "treated", "treated"))
 groups <- c(rep("New_Brunswick", times = 4), rep("Louisiana", times = 4))
y <- DGEList(counts = gene_counts, group = groups)


Loading required package: limma



In [8]:
low_expression_rows <- filterByExpr(y)
filtered_y <- y[low_expression_rows, , keep.lib.sizes = FALSE]


calculate the normalization factor by taking into the account the overall size of the library. 
We must normalize how much expression of any one gene is happening by comparing against how much all genees are expressed within the library

In [9]:
filtered_y <- calcNormFactors(filtered_y)
design <- model.matrix(~ groups)


In [ ]:
Estimate the dispersion

In [10]:
filtered_y <- estimateDisp(filtered_y, design)


In [11]:
# Fit the quasi-likelihood negative binomial generalized log-linear model
fit <- glmQLFit(filtered_y, design)

# Conduct the quasi-likelihood F-test for the second coefficient (e.g., treated vs control)
qlf <- glmQLFTest(fit, coef = 2)

# View top differentially expressed genes
topTags(qlf)

# Extract a full table of results
res <- topTags(qlf, n = 20)$table




,logFC,logCPM,F,PValue,FDR
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
gene-LOC111130444,-7.337925,3.405214,341.72945,1.024666e-08,0.0002313491
gene-LOC144622451,4.122710,4.619771,153.39117,3.868668e-07,0.0028019603
gene-LOC111132320,7.003352,2.897777,149.51381,4.336107e-07,0.0028019603
gene-LOC111104759,4.601623,5.940917,145.03668,4.964054e-07,0.0028019603
gene-LOC111136420,6.520857,7.366913,122.00804,1.066390e-06,0.0043701313
gene-LOC111120559,-2.093746,7.990116,119.66353,1.161342e-06,0.0043701313
gene-LOC111120017,-7.061463,3.050806,113.70851,1.452739e-06,0.0046857064
gene-LOC144618722,-6.237235,2.849338,107.85815,1.829932e-06,0.0051645247
gene-LOC144627087,3.267527,5.204361,103.84509,2.158534e-06,0.0054150435


,logFC,logCPM,F,PValue,FDR
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
gene-LOC111130444,-7.337925,3.405214,341.72945,1.024666e-08,0.0002313491
gene-LOC144622451,4.122710,4.619771,153.39117,3.868668e-07,0.0028019603
gene-LOC111132320,7.003352,2.897777,149.51381,4.336107e-07,0.0028019603
gene-LOC111104759,4.601623,5.940917,145.03668,4.964054e-07,0.0028019603
gene-LOC111136420,6.520857,7.366913,122.00804,1.066390e-06,0.0043701313
gene-LOC111120559,-2.093746,7.990116,119.66353,1.161342e-06,0.0043701313
gene-LOC111120017,-7.061463,3.050806,113.70851,1.452739e-06,0.0046857064
gene-LOC144618722,-6.237235,2.849338,107.85815,1.829932e-06,0.0051645247
gene-LOC144627087,3.267527,5.204361,103.84509,2.158534e-06,0.0054150435


In [12]:
# Now we will get the gaf file in order to map the gene ontology names
system("wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/053/477/285/GCF_053477285.1_ASM5347728v1/GCF_053477285.1-RS_2025_12_gene_ontology.gaf.gz")

In [13]:
system("gunzip GCF_053477285.1-RS_2025_12_gene_ontology.gaf.gz")

In [15]:
system("conda install -c bioconductor mgsa")

In [16]:
gaf <- read.table("GCF_053477285.1-RS_2025_12_gene_ontology.gaf", 
                   sep="\t", comment.char="!", header=FALSE, quote="")

In [18]:
head(gaf)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17
,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<int>,<chr>,<lgl>,<lgl>
1,NCBIGene,111099029,LOC111099029,enables,GO:0005261,PMID:30032202,IEA,PANTHER:PTN007624244,F,transient receptor potential cation channel subfamily M member-like 2,NA,protein,taxon:6565,20260105,RefSeq,NA,NA
2,NCBIGene,111099029,LOC111099029,located_in,GO:0005886,PMID:30032202,IEA,PANTHER:PTN007624244,C,transient receptor potential cation channel subfamily M member-like 2,NA,protein,taxon:6565,20260105,RefSeq,NA,NA
3,NCBIGene,111099029,LOC111099029,involved_in,GO:0030001,PMID:30032202,IEA,PANTHER:PTN007624244,P,transient receptor potential cation channel subfamily M member-like 2,NA,protein,taxon:6565,20260105,RefSeq,NA,NA
4,NCBIGene,111099029,LOC111099029,involved_in,GO:0098655,PMID:30032202,IEA,PANTHER:PTN007624244,P,transient receptor potential cation channel subfamily M member-like 2,NA,protein,taxon:6565,20260105,RefSeq,NA,NA
5,NCBIGene,111099032,LOC111099032,enables,GO:0004930,PMID:22301074,IEA,InterPro:IPR000276,F,uncharacterized LOC111099032,NA,protein,taxon:6565,20260105,RefSeq,NA,NA
6,NCBIGene,111099032,LOC111099032,enables,GO:0004930,PMID:30032202,IEA,PANTHER:PTN009087238,F,uncharacterized LOC111099032,NA,protein,taxon:6565,20260105,RefSeq,NA,NA
